# Evalúa tu modelo base

**Sesión 4 · IA Intermedio · EstudIA 2026**

En la sesión 3 probaste tu modelo base con unos pocos ejemplos. Hoy lo **medimos**: con ejemplos que
tienen su etiqueta, calculamos cuántos acierta y lo comparamos con una **línea base**, la regla más
simple posible: responder siempre la clase más frecuente. Si tu modelo no le gana, todavía no sirve.

| Camino | Para quién | Qué haces |
|---|---|---|
| **A. Tu proyecto** | Tienes un modelo base y de 20 a 50 ejemplos con su etiqueta | Mides tu modelo con tus ejemplos |
| **B. Práctica** | Todavía no tienes modelo o ejemplos | Mides dos modelos zero-shot: texto o imagen (o los dos) |

Cuando termines, **entrega el notebook** desde la última celda. La entrega muestra que mediste un
modelo de Hugging Face. No importa si el modelo acierta.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU: los modelos de este
notebook funcionan sin ella.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s04")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from transformers import pipeline

DISPOSITIVO = 0 if torch.cuda.is_available() else -1
print("Listo.", f"GPU: {torch.cuda.get_device_name(0)}." if DISPOSITIVO == 0 else "Sin GPU: los modelos de este notebook funcionan igual.")

In [ ]:
# @title Funciones auxiliares: descargar archivos, escribir números y evaluar
def descargar(nombre):
    """Baja un archivo de la carpeta pública del curso (solo la primera vez) y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def evaluar(verdad, prediccion, titulo="Tu modelo"):
    """Compara las predicciones con las etiquetas reales y con la línea base de la clase más frecuente."""
    verdad, prediccion = [str(v) for v in verdad], [str(p) for p in prediccion]
    clases = sorted(set(verdad))
    frecuente = max(clases, key=verdad.count)
    base = [frecuente] * len(verdad)
    def balanceada(p):
        return float(np.mean([np.mean([pi == c for vi, pi in zip(verdad, p) if vi == c]) for c in clases]))
    exac = np.mean([v == p for v, p in zip(verdad, prediccion)])
    print(f"{titulo} · {len(verdad)} ejemplos, {len(clases)} clases")
    print(f"  Exactitud:            {pct(exac)}   (línea base “siempre {frecuente}”: {pct(np.mean([v == frecuente for v in verdad]))})")
    print(f"  Exactitud balanceada: {pct(balanceada(prediccion))}   (línea base: {pct(balanceada(base))})")
    print("  Recall por clase:")
    for c in clases:
        n = verdad.count(c)
        bien = sum(v == c and p == c for v, p in zip(verdad, prediccion))
        print(f"    {c:24s} {bien} de {n} ({pct(bien / n, 0)})")
    todas = clases + sorted(set(prediccion) - set(clases))
    fig, eje = plt.subplots(figsize=(1.1 * len(todas) + 2.5, 0.9 * len(clases) + 1.8))
    ConfusionMatrixDisplay(confusion_matrix(verdad, prediccion, labels=todas), display_labels=todas).plot(
        ax=eje, cmap="Purples", colorbar=False, xticks_rotation=45)
    eje.set_xlabel("Lo que respondió"); eje.set_ylabel("La etiqueta real"); eje.set_title(titulo)
    eje.set_ylim(len(clases) - 0.5, -0.5)   # solo las filas de las clases reales
    plt.tight_layout(); plt.show()

---
## A. Tu proyecto

### A1. Tus ejemplos

Escríbelos en `EJEMPLOS`, cada uno como `("entrada", "etiqueta")`, o sube un archivo `ejemplos.csv` con
dos columnas, `entrada` y `etiqueta` (ícono de carpeta, a la izquierda de Colab). Si tu proyecto es de
imágenes, la entrada es el nombre del archivo que subiste, por ejemplo `"hoja_01.jpg"`.

Sin datos personales. Usa las mismas etiquetas que quieres que tu proyecto entregue.

In [ ]:
EJEMPLOS = [
    ("Escribe aquí un ejemplo real de tu proyecto.", "etiqueta_1"),
    ("Y otro, con otra etiqueta.", "etiqueta_2"),
]

if os.path.exists("ejemplos.csv"):
    tabla = pd.read_csv("ejemplos.csv")
    EJEMPLOS = list(zip(tabla["entrada"].astype(str), tabla["etiqueta"].astype(str)))
mis_ejemplos = not EJEMPLOS[0][0].startswith("Escribe aquí")
if mis_ejemplos:
    entradas = [e for e, _ in EJEMPLOS]
    verdad = [str(et) for _, et in EJEMPLOS]
    print(f"{len(EJEMPLOS)} ejemplos. Etiquetas:", pd.Series(verdad).value_counts().to_dict())
else:
    print("Camino A: escribe tus ejemplos en EJEMPLOS (o sube ejemplos.csv) y vuelve a ejecutar. "
          "Si vas por el camino B, sigue a la parte B.")

### A2. Tu modelo

Escribe el nombre de tu modelo en `MODELO`. La celda imprime la tarea del modelo y, si es un
clasificador, las etiquetas que sabe responder. Si `pipeline` dice que no puede adivinar la tarea,
escríbela en `TAREA` (la ves en la página del modelo en Hugging Face, arriba del nombre).

In [ ]:
MODELO = "dueño/nombre-del-modelo"   # cámbialo, por ejemplo: "cardiffnlp/twitter-xlm-roberta-base-sentiment"
TAREA = None   # si pipeline no adivina la tarea, escríbela, por ejemplo: "text-classification"

modelo = None
if MODELO.startswith("dueño/"):
    print("Escribe el nombre de tu modelo en MODELO y vuelve a ejecutar esta celda.")
else:
    modelo = pipeline(task=TAREA, model=MODELO, device=DISPOSITIVO)
    print("Tarea del modelo:", modelo.task)
    if "zero-shot" not in modelo.task and hasattr(modelo.model.config, "id2label"):
        print("Etiquetas que sabe responder:", list(modelo.model.config.id2label.values()))

### A3. Sus etiquetas y las tuyas

El modelo responde con **sus** etiquetas, que casi nunca son las tuyas. Hay dos casos:

- **Modelo zero-shot** (`zero-shot-classification` o `zero-shot-image-classification`): le damos tus
  etiquetas y elige entre ellas. No tienes que hacer nada más.
- **Clasificador con etiquetas propias**: escribe en `TRADUCCION` a qué etiqueta tuya corresponde cada
  etiqueta del modelo. Las que no traduzcas quedan como `"(otra)"` y cuentan como error.

Si ninguna etiqueta del modelo corresponde a las tuyas, ese modelo necesita una **cabeza nueva**: es
justo lo que hicimos hoy en la demostración.

In [ ]:
TRADUCCION = {
    # "etiqueta del modelo": "tu etiqueta",
}

if modelo is not None and mis_ejemplos:
    clases = sorted(set(verdad))
    if "zero-shot" in modelo.task:
        respuestas = modelo(entradas, candidate_labels=clases)
        respuestas = [r if isinstance(r, list) else [{"label": l, "score": s} for l, s in zip(r["labels"], r["scores"])]
                      for r in respuestas]
        prediccion = [r[0]["label"] for r in respuestas]
    else:
        respuestas = modelo(entradas)
        prediccion = [TRADUCCION.get((r[0] if isinstance(r, list) else r)["label"], "(otra)") for r in respuestas]
    evaluar(verdad, prediccion, MODELO)
    print("\nErrores:")
    for e, v, p in zip(entradas, verdad, prediccion):
        if v != p:
            print(f"  ✗ {str(e)[:70]} · era {v}, respondió {p}")

**Lo que observé** (doble clic aquí y responde en dos o tres líneas):

- ¿Tu modelo le gana a la línea base? ¿Por cuánto, en exactitud y en exactitud balanceada?
- ¿Qué clase le cuesta más? ¿Es la que más le costaría equivocarse a quien use tu modelo?
- ¿Te alcanza con este modelo tal cual, o habría que ajustarlo? Si hay que ajustarlo, ¿cuántos ejemplos
  etiquetados más crees que necesitarías?

---
## B. Práctica

Dos tareas con su etiqueta. Medimos modelos **zero-shot**: no conocen nuestras clases de antemano;
nosotros escribimos las etiquetas y el modelo elige la que mejor encaja. En cada tarea completas una parte
del código; si te atascas, abre la **pista** que está debajo de la celda.

### B1. Texto: ¿de qué tema es cada mensaje?

Treinta mensajes ciudadanos, cinco por tema, con
[`MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli`](https://huggingface.co/MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli)
(licencia MIT), el mismo modelo de la sesión 3.

In [ ]:
MENSAJES = [
    ("El semáforo de la 80 con la 33 lleva tres días apagado y se forman unos trancones terribles.", "movilidad"),
    ("El metro iba tan lleno esta mañana que no pude subirme en tres trenes.", "movilidad"),
    ("Los buses de la ruta del barrio dejaron de pasar después de las ocho de la noche.", "movilidad"),
    ("Hay un hueco enorme en la vía principal y ya se han caído dos motociclistas.", "movilidad"),
    ("No hay señalización en la nueva ciclorruta y los carros se parquean encima.", "movilidad"),
    ("Anoche robaron dos celulares en el paradero del parque y no había ningún policía.", "seguridad"),
    ("Pusieron cámaras en la ciclorruta, pero siguen robando bicicletas.", "seguridad"),
    ("En la esquina de la cancha venden droga a plena luz del día.", "seguridad"),
    ("Las luces del puente peatonal están dañadas y da miedo pasar de noche.", "seguridad"),
    ("Unos muchachos están cobrando vacuna a los tenderos de la cuadra.", "seguridad"),
    ("En el centro de salud del barrio no hay citas para medicina general hasta el próximo mes.", "salud"),
    ("Llevo tres horas esperando en urgencias con mi mamá y nadie nos atiende.", "salud"),
    ("La farmacia de la EPS no tiene la insulina desde hace dos semanas.", "salud"),
    ("Hay muchos casos de dengue en el sector y nadie ha venido a fumigar.", "salud"),
    ("Necesitamos una jornada de vacunación para los niños de la vereda.", "salud"),
    ("La escuela de la vereda no tiene profesor de matemáticas desde marzo.", "educación"),
    ("Al colegio no ha llegado el refrigerio de los estudiantes en todo el mes.", "educación"),
    ("Los computadores de la sala de informática del colegio llevan un año dañados.", "educación"),
    ("No hay cupos en el preescolar del barrio y tengo que trabajar.", "educación"),
    ("El techo del salón se cae a pedazos cuando llueve y los niños reciben clase así.", "educación"),
    ("Hace una semana se fue el agua en toda la cuadra y nadie da razón.", "servicios públicos"),
    ("La factura de la luz llegó el doble de lo normal sin ninguna explicación.", "servicios públicos"),
    ("Hay una fuga de gas en el contador del edificio y huele muy fuerte.", "servicios públicos"),
    ("El alcantarillado se rebosa cada vez que llueve y el agua entra a las casas.", "servicios públicos"),
    ("Se cayó un cable de energía en el andén y está echando chispas.", "servicios públicos"),
    ("Están quemando basura al lado de la quebrada y el humo llega a todas las casas.", "medio ambiente"),
    ("Talaron los árboles del parque sin ningún permiso.", "medio ambiente"),
    ("Una fábrica está botando aguas negras al río.", "medio ambiente"),
    ("La calidad del aire está pésima y los niños tienen tos todo el día.", "medio ambiente"),
    ("Dejan escombros y colchones viejos en la orilla de la quebrada.", "medio ambiente"),
]
# Lo que le damos al modelo para cada tema: una frase que lo describe.
# Completa las dos que faltan: cambia cada ... por una frase entre comillas.
ETIQUETAS = {"movilidad": "tránsito y transporte", "seguridad": "seguridad y delincuencia",
             "salud": ..., "educación": "educación y colegios",
             "servicios públicos": "agua, luz o gas", "medio ambiente": ...}

if ... in ETIQUETAS.values():
    print("Completa las frases de “salud” y “medio ambiente” en ETIQUETAS y vuelve a ejecutar. "
          "Si te atascas, abre la pista de abajo.")
else:
    clasificador_texto = pipeline("zero-shot-classification", model="MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli",
                                  device=DISPOSITIVO)
    respuestas = clasificador_texto([m for m, _ in MENSAJES], candidate_labels=list(ETIQUETAS.values()),
                                    hypothesis_template="Este mensaje trata sobre {}.")
    tema_de = {v: k for k, v in ETIQUETAS.items()}
    evaluar([t for _, t in MENSAJES], [tema_de[r["labels"][0]] for r in respuestas], "Zero-shot de texto")

<details><summary>Pista</summary>

Mira las otras cuatro frases: nombran el tema con dos o tres palabras concretas, de las que aparecerían en un mensaje
sobre ese tema (“tránsito y transporte”, “agua, luz o gas”). Para salud sirve `"salud y hospitales"`, y para medio
ambiente, `"medio ambiente y contaminación"`. Van entre comillas, en lugar de los `...`.
</details>

Prueba algo y vuelve a ejecutar: cambia cada frase de `ETIQUETAS` por el nombre corto del tema (por
ejemplo, `"movilidad": "movilidad"`) o por otras frases, o agrega mensajes difíciles. ¿Qué pasa con la
exactitud? ¿Cuál clase mejora y cuál empeora?

### B2. Imagen: ¿qué uso tiene este suelo?

Treinta imágenes satelitales de 64 × 64 píxeles, tres por clase, con
[`google/siglip2-base-patch16-224`](https://huggingface.co/google/siglip2-base-patch16-224) (licencia
Apache 2.0) y etiquetas en español. En la demostración, el modelo ajustado con 2.000 imágenes acertó
casi todas.

In [ ]:
practica = np.load(descargar("practica_satelite.npz"))
NOMBRES = {"autopista": "una autopista", "bosque": "un bosque", "cultivo_anual": "un cultivo anual",
           "cultivo_permanente": "un cultivo permanente con árboles frutales", "industrial": "una zona industrial",
           "mar_o_lago": "un mar o un lago", "pastizal": "un pastizal para ganado", "residencial": "un barrio de casas",
           "rio": "un río", "vegetacion_herbacea": "vegetación silvestre"}
clasificador_imagen = pipeline("zero-shot-image-classification", model="google/siglip2-base-patch16-224",
                               device=DISPOSITIVO)
imagenes = [Image.fromarray(x).resize((224, 224), Image.BICUBIC) for x in practica["imagenes"]]
respuestas = clasificador_imagen(imagenes, candidate_labels=list(NOMBRES.values()),
                                 hypothesis_template="Una foto satelital de {}.")
print("Las dos opciones más probables para la primera imagen:", respuestas[0][:2])
de_vuelta = {v: k for k, v in NOMBRES.items()}   # de "un bosque" a "bosque"

# Completa: para cada respuesta r, toma la opción más probable y conviértela en nuestra etiqueta con de_vuelta.
prediccion = ...

if prediccion is ...:
    print("Completa la línea de prediccion y vuelve a ejecutar. Si te atascas, abre la pista de abajo.")
else:
    evaluar(practica["etiquetas"].tolist(), prediccion, "Zero-shot de imagen")

<details><summary>Pista</summary>

Cada respuesta es una lista de opciones, de la más probable a la menos probable: `r[0]` es la primera y `r[0]["label"]`
es su texto, por ejemplo `"un bosque"`. `de_vuelta` lo convierte en nuestra etiqueta. La línea completa:

```python
prediccion = [de_vuelta[r[0]["label"]] for r in respuestas]
```
</details>

**Lo que observé** (doble clic aquí y responde): ¿le ganan los dos modelos a la línea base? ¿Qué clases
confunden? ¿Qué te dice eso sobre cuándo basta un modelo zero-shot y cuándo hay que ajustar?

**Fuente de las imágenes:** EuroSAT (Helber y otros, 2019), licencia MIT, imágenes de Copernicus Sentinel-2 (ESA).

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook
con sus resultados y lo envía a la plataforma. Te pide tu clave la primera vez.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la
plataforma, en la actividad “Entrega: evalúa tu modelo base”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error, urllib.request

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: evalúa tu modelo base”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s04", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()